# Blog figures for the explainability report

This notebook generates the additional explanatory figures used in the final HTML report. It does not require GPU execution; it reads existing CSV outputs from `outputs/reports/` and writes figures to `outputs/figures/` and `docs/`.

In [1]:

from pathlib import Path
import shutil
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures'
DOCS_DIR = PROJECT_ROOT / 'docs'
REPORT_DIR = PROJECT_ROOT / 'outputs' / 'reports'
DOC_ASSET_DIR = DOCS_DIR / 'assets' / 'xai-report'
XAI_EXAMPLE_FIG = FIG_DIR / 'xai_method_comparison_notebook.png'
ADVANCED_FIGURE_DIR = FIG_DIR / 'advanced_attribution_audit_notebook'
FAITHFULNESS_FIGURES = {
    'gradcam': ADVANCED_FIGURE_DIR / 'gradcam_deletion_insertion.png',
    'integrated_gradients': ADVANCED_FIGURE_DIR / 'integrated_gradients_deletion_insertion.png',
}
FIG_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
DOC_ASSET_DIR.mkdir(parents=True, exist_ok=True)
required_blog_sources = [XAI_EXAMPLE_FIG, *FAITHFULNESS_FIGURES.values()]
missing_blog_sources = [path for path in required_blog_sources if not path.exists()]
if missing_blog_sources:
    raise FileNotFoundError(f'Run the local-attribution cell in 01_data_baseline_xai.ipynb and the advanced attribution audit in 02_stress_concepts_tcav.ipynb before exporting blog figures: {missing_blog_sources}')
shutil.copy2(XAI_EXAMPLE_FIG, DOC_ASSET_DIR / 'correct-attributions.png')
shutil.copy2(FAITHFULNESS_FIGURES['gradcam'], DOC_ASSET_DIR / 'advanced-gradcam-deletion.png')
shutil.copy2(FAITHFULNESS_FIGURES['integrated_gradients'], DOC_ASSET_DIR / 'advanced-ig-deletion.png')

plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 180,
    'font.size': 10,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

PALETTE = {
    'navy': '#18324a',
    'teal': '#2a9d8f',
    'amber': '#e9a23b',
    'red': '#d95f5f',
    'blue': '#4f7cac',
    'soft': '#f4f8f7',
    'line': '#cfd8dc',
    'ink': '#263238',
    'muted': '#61717a',
}

def save_status_figure(path, title, message):
    fig, ax = plt.subplots(figsize=(9, 4.8))
    fig.patch.set_facecolor('white')
    ax.axis('off')
    ax.text(0.5, 0.62, title, transform=ax.transAxes, ha='center', va='center', fontsize=20, fontweight='bold', color=PALETTE['navy'])
    ax.text(0.5, 0.38, message, transform=ax.transAxes, ha='center', va='center', fontsize=12, color=PALETTE['muted'], wrap=True)
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    print(f'saved status figure {path}')

def save_svg(path, body, width=1320, height=420):
    svg = f'''<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" role="img">
  <rect width="100%" height="100%" rx="22" fill="#f7faf9"/>
  {body}
</svg>'''
    path.write_text(svg, encoding='utf-8')
    print(f'saved {path}')

def svg_text(x, y, text, size=22, weight=600, color='#18324a', anchor='middle', width=24):
    lines = textwrap.wrap(text, width=width)
    out = []
    for i, line in enumerate(lines):
        out.append(f'<text x="{x}" y="{y + i * (size + 5)}" font-family="Inter, Arial, sans-serif" font-size="{size}" font-weight="{weight}" fill="{color}" text-anchor="{anchor}">{line}</text>')
    return '\n'.join(out)

def svg_box(x, y, w, h, title, subtitle, fill='#ffffff', stroke='#cfd8dc'):
    return f'''
  <rect x="{x}" y="{y}" width="{w}" height="{h}" rx="18" fill="{fill}" stroke="{stroke}" stroke-width="2"/>
  {svg_text(x + w/2, y + 38, title, size=21, weight=700, width=18)}
  {svg_text(x + w/2, y + 82, subtitle, size=15, weight=500, color='#61717a', width=18)}
'''

def svg_arrow(x1, y1, x2, y2, color='#2a9d8f'):
    return f'<path d="M{x1} {y1} C {(x1+x2)/2} {y1}, {(x1+x2)/2} {y2}, {x2} {y2}" fill="none" stroke="{color}" stroke-width="4" marker-end="url(#arrow)"/>'

pipeline_body = '''
  <defs><marker id="arrow" markerWidth="12" markerHeight="12" refX="10" refY="6" orient="auto"><path d="M2,2 L10,6 L2,10 Z" fill="#2a9d8f"/></marker></defs>
  <text x="660" y="52" font-family="Inter, Arial, sans-serif" font-size="30" font-weight="700" fill="#18324a" text-anchor="middle">Explainability audit pipeline</text>
'''
boxes = [
    (40, 120, 185, 150, 'Image input', 'AwA2 RGB image 224 x 224 tensor'),
    (260, 120, 185, 150, 'ResNet50', 'feature extractor logits + softmax'),
    (480, 120, 185, 150, 'Attribution', 'Grad-CAM, IG'),
    (700, 120, 185, 150, 'Stress test', 'background noise, color shift, swap'),
    (920, 120, 185, 150, 'Metrics', 'IoU, Spearman, deletion, insertion'),
    (1140, 120, 140, 150, 'Concepts', 'TCAV + CBM'),
]
for b in boxes:
    pipeline_body += svg_box(*b)
for a in [(225, 195, 260, 195), (445, 195, 480, 195), (665, 195, 700, 195), (885, 195, 920, 195), (1105, 195, 1140, 195)]:
    pipeline_body += svg_arrow(*a)
pipeline_body += '<text x="660" y="340" font-family="Inter, Arial, sans-serif" font-size="20" font-weight="600" fill="#263238" text-anchor="middle">A heatmap becomes credible only when it survives behavioral, stability and concept-level checks.</text>'
save_svg(DOCS_DIR / 'blog_project_pipeline.svg', pipeline_body, 1320, 400)

cols = [
    ('Activation localization', 'Grad-CAM', 'Where do final feature maps support the class?', '#e8f4f2'),
    ('Gradient path', 'Integrated Gradients', 'How does the score change from baseline to image?', '#eef3fb'),
    ('Concept direction', 'TCAV', 'Which semantic directions affect the class?', '#fff6e5'),
    ('Concept mediation', 'Concept Bottleneck', 'Can predictions pass through human-readable attributes?', '#fbeeee'),
]
body = '<text x="660" y="50" font-family="Inter, Arial, sans-serif" font-size="30" font-weight="700" fill="#18324a" text-anchor="middle">Taxonomy of explanation methods used in the project</text>'
for i, (title, methods, question, fill) in enumerate(cols):
    x = 45 + i * 315
    body += svg_box(x, 105, 280, 220, title, methods, fill=fill)
    body += svg_text(x + 140, 252, question, size=15, weight=500, color='#263238', width=29)
save_svg(DOCS_DIR / 'blog_method_taxonomy.svg', body, 1320, 380)

steps = [
    ('1', 'Visual plausibility', 'Does the heatmap cover a meaningful region?'),
    ('2', 'Spatial stability', 'Does it remain stable under controlled changes?'),
    ('3', 'Behavioral faithfulness', 'Do highlighted pixels affect the score?'),
    ('4', 'Concept checks', 'Do concepts align with the prediction?'),
    ('5', 'Concept alignment', 'Do concepts support the same interpretation?'),
]
body = '<defs><marker id="arrow" markerWidth="12" markerHeight="12" refX="10" refY="6" orient="auto"><path d="M2,2 L10,6 L2,10 Z" fill="#e9a23b"/></marker></defs>'
body += '<text x="660" y="48" font-family="Inter, Arial, sans-serif" font-size="30" font-weight="700" fill="#18324a" text-anchor="middle">Evidence ladder for a reliable explainability claim</text>'
for i, (num, title, subtitle) in enumerate(steps):
    x = 60 + i * 245
    y = 120 + i * 18
    body += f'<circle cx="{x+35}" cy="{y+35}" r="34" fill="#2a9d8f"/><text x="{x+35}" y="{y+44}" font-family="Inter, Arial, sans-serif" font-size="26" font-weight="700" fill="#ffffff" text-anchor="middle">{num}</text>'
    body += svg_box(x+82, y, 155, 100, title, subtitle, fill='#ffffff')
    if i < len(steps) - 1:
        body += svg_arrow(x + 235, y + 50, x + 275, y + 68, color='#e9a23b')
save_svg(DOCS_DIR / 'blog_evidence_ladder.svg', body, 1320, 430)

advanced = pd.read_csv(REPORT_DIR / 'advanced_attribution_audit_notebook_summary.csv')
PHASE5_CSV = REPORT_DIR / 'phase5_saliency_metrics_notebook.csv'
stress_metrics = pd.read_csv(PHASE5_CSV)
IOU_COLUMN = next((column for column in stress_metrics if column.startswith('iou_top_')), None)
if IOU_COLUMN is None:
    raise ValueError('Stress metrics CSV is missing an iou_top_* column.')
required_confidence_columns = {'original_target_probability', 'perturbed_target_probability', 'confidence_delta', 'confidence_drop'}
missing_confidence_columns = required_confidence_columns.difference(stress_metrics.columns)
if missing_confidence_columns:
    raise ValueError(f'Regenerate the stress metrics CSV; missing corrected confidence columns: {sorted(missing_confidence_columns)}')
for column in ['original_confidence', 'perturbed_confidence', *sorted(required_confidence_columns), IOU_COLUMN, 'spearman']:
    stress_metrics[column] = pd.to_numeric(stress_metrics[column], errors='coerce')
stress_metrics['prediction_changed'] = stress_metrics['prediction_changed'].astype(str).str.lower().eq('true')
stress_metrics['saliency_drift'] = (1.0 - stress_metrics[IOU_COLUMN]) + (1.0 - stress_metrics['spearman']) / 2.0
stress_metrics['prediction_transition'] = stress_metrics['original_prediction'].astype(str) + ' -> ' + stress_metrics['perturbed_prediction'].astype(str)
stress = (
    stress_metrics.groupby(['xai_method', 'perturbation'], as_index=False)
    .agg(
        prediction_change_rate=('prediction_changed', 'mean'),
        mean_iou=(IOU_COLUMN, 'mean'),
    )
)
unstable = stress_metrics.sort_values('saliency_drift', ascending=False).head(12)
method_names = {'gradcam': 'Grad-CAM', 'integrated_gradients': 'Integrated\nGradients'}
maintained_methods = list(method_names)
advanced = advanced[advanced['method'].isin(maintained_methods)].copy()
advanced['label'] = advanced['method'].map(method_names)
fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.2))
fig.patch.set_facecolor('white')

ax = axes[0, 0]
y = np.arange(len(advanced))
ax.barh(y, advanced['mean_animal_saliency_ratio'], color=PALETTE['teal'], label='animal')
ax.barh(y, advanced['mean_background_saliency_ratio'], left=advanced['mean_animal_saliency_ratio'], color=PALETTE['amber'], label='background')
ax.set_yticks(y, advanced['label'])
ax.set_xlim(0, 1)
ax.set_title('Where the saliency mass is located', pad=24)
ax.set_xlabel('fraction of total saliency')
for i, row in advanced.iterrows():
    ax.text(row['mean_animal_saliency_ratio'] / 2, i, f"{row['mean_animal_saliency_ratio']:.2f}", va='center', ha='center', color='white', fontweight='bold')
    ax.text(row['mean_animal_saliency_ratio'] + row['mean_background_saliency_ratio'] / 2, i, f"{row['mean_background_saliency_ratio']:.2f}", va='center', ha='center', color=PALETTE['navy'], fontweight='bold')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=False)
ax.grid(axis='x', alpha=0.18)

ax = axes[0, 1]
x = np.arange(len(advanced))
w = 0.38
ax.bar(x - w/2, advanced['mean_sensitivity_iou_top20'], width=w, color=PALETTE['blue'], label='IoU')
ax.bar(x + w/2, advanced['mean_sensitivity_spearman'], width=w, color=PALETTE['teal'], label='Spearman')
ax.set_xticks(x, advanced['label'])
ax.set_ylim(0, 1)
ax.set_title('Sensitivity stability under input noise')
ax.set_ylabel('higher is more stable')
for xpos, vals in [(x - w/2, advanced['mean_sensitivity_iou_top20']), (x + w/2, advanced['mean_sensitivity_spearman'])]:
    for xx, val in zip(xpos, vals):
        ax.text(xx, val + 0.025, f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.18)

ax = axes[1, 0]
ax.bar(x - w/2, advanced['mean_deletion_auc'], width=w, color=PALETTE['red'], label='Deletion AUC')
ax.bar(x + w/2, advanced['mean_insertion_auc'], width=w, color=PALETTE['teal'], label='Insertion AUC')
ax.set_xticks(x, advanced['label'])
ax.set_ylim(0, max(advanced['mean_insertion_auc'].max(), advanced['mean_deletion_auc'].max()) * 1.35)
ax.set_title('Behavioral faithfulness curves')
ax.set_ylabel('AUC')
for xpos, vals in [(x - w/2, advanced['mean_deletion_auc']), (x + w/2, advanced['mean_insertion_auc'])]:
    for xx, val in zip(xpos, vals):
        ax.text(xx, val + 0.01, f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.18)

ax = axes[1, 1]
bs = stress[stress['perturbation'].eq('background_swap') & stress['xai_method'].isin(maintained_methods)].copy()
bs['label'] = bs['xai_method'].map(method_names)
xx = np.arange(len(bs))
ax.bar(xx - w/2, bs['mean_iou'], width=w, color=PALETTE['blue'], label='background-swap IoU')
ax.bar(xx + w/2, bs['prediction_change_rate'], width=w, color=PALETTE['amber'], label='prediction change rate')
ax.set_xticks(xx, bs['label'])
ax.set_ylim(0, 1)
ax.set_title('Background-swap stress test')
ax.set_ylabel('rate / overlap')
for xpos, vals in [(xx - w/2, bs['mean_iou']), (xx + w/2, bs['prediction_change_rate'])]:
    for xxv, val in zip(xpos, vals):
        ax.text(xxv, val + 0.025, f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.18)

for ax in axes.ravel():
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_color('#d7dee2')
    ax.spines['bottom'].set_color('#d7dee2')

fig.suptitle('Quantitative evidence: saliency location, stability and faithfulness', fontsize=15, fontweight='bold', color=PALETTE['navy'])
fig.tight_layout(rect=[0, 0, 1, 0.95])
out = FIG_DIR / 'blog_metric_summary.png'
fig.savefig(out, bbox_inches='tight')
shutil.copy2(out, DOC_ASSET_DIR / 'metric-summary.png')
plt.close(fig)
print(f'saved {out}')

case_pool = stress_metrics[stress_metrics['perturbation'].eq('background_swap')].copy()
changed_pool = case_pool[case_pool['prediction_changed']].copy()
case_has_transition = not changed_pool.empty
selection_pool = changed_pool if case_has_transition else case_pool
case_index = selection_pool.groupby('index')['saliency_drift'].mean().idxmax()
case = case_pool[case_pool['index'].eq(case_index) & case_pool['xai_method'].isin(['gradcam', 'integrated_gradients'])].copy()
if case.empty:
    case = selection_pool[selection_pool['index'].eq(case_index)].head(2).copy()
case['label'] = case['xai_method'].map({'gradcam': 'Grad-CAM', 'integrated_gradients': 'IG'}).fillna(case['xai_method'])
base = case.iloc[0]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1.1, 1, 1.2]})
fig.patch.set_facecolor('white')

ax = axes[0]
ax.axis('off')
ax.set_title('Single unstable example', color=PALETTE['navy'], fontweight='bold')
summary = [
    ('true class', str(base['true_class'])),
    ('original prediction', f"{base['original_prediction']} ({base['original_confidence']:.2f})"),
    ('after background swap', f"{base['perturbed_prediction']} ({base['perturbed_confidence']:.2f})"),
    ('transition', str(base['prediction_transition'])),
]
for i, (k, v) in enumerate(summary):
    y0 = 0.82 - i * 0.19
    patch = FancyBboxPatch((0.02, y0 - 0.075), 0.96, 0.13, boxstyle='round,pad=0.018,rounding_size=0.02', transform=ax.transAxes, fc=PALETTE['soft'], ec=PALETTE['line'])
    ax.add_patch(patch)
    ax.text(0.07, y0 + 0.018, k.upper(), transform=ax.transAxes, fontsize=8, color=PALETTE['muted'], fontweight='bold')
    ax.text(0.07, y0 - 0.035, v, transform=ax.transAxes, fontsize=11, color=PALETTE['ink'], fontweight='bold')

ax = axes[1]
vals = [base['original_target_probability'], base['perturbed_target_probability']]
ax.bar(['original', 'perturbed'], vals, color=[PALETTE['teal'], PALETTE['amber']])
ax.set_ylim(0, 1)
ax.set_title('Confidence drop')
ax.set_ylabel('fixed saliency-target probability')
for i, val in enumerate(vals):
    ax.text(i, val + 0.03, f'{val:.2f}', ha='center', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.18)

ax = axes[2]
labels = list(case['label'])
xpos = np.arange(len(case))
ax.bar(xpos - 0.18, case[IOU_COLUMN], width=0.36, color=PALETTE['blue'], label='Top-20 IoU')
ax.bar(xpos + 0.18, case['spearman'], width=0.36, color=PALETTE['teal'], label='Spearman')
ax.set_xticks(xpos, labels)
ax.set_ylim(0, 1)
ax.set_title('Explanation stability after background swap')
ax.set_ylabel('higher is more stable')
for xxv, val in zip(xpos - 0.18, case[IOU_COLUMN]):
    ax.text(xxv, val + 0.03, f'{val:.2f}', ha='center', fontsize=9)
for xxv, val in zip(xpos + 0.18, case['spearman']):
    ax.text(xxv, val + 0.03, f'{val:.2f}', ha='center', fontsize=9)
ax.legend(frameon=False, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.18)

case_title = ('Case study: a background change switched the class and moved the explanation' if case_has_transition else 'Case study: a background change moved the explanation')
fig.suptitle(case_title, fontsize=15, fontweight='bold', color=PALETTE['navy'])
fig.tight_layout(rect=[0, 0, 1, 0.92])
out = FIG_DIR / 'blog_case_study_summary.png'
fig.savefig(out, bbox_inches='tight')
shutil.copy2(out, DOC_ASSET_DIR / 'case-summary.png')
plt.close(fig)
print(f'saved {out}')

TCAV_CSV = REPORT_DIR / 'phase7_tcav_scores_notebook.csv'
CBM_SUMMARY_CSV = REPORT_DIR / 'phase8_cbm_summary_notebook.csv'
IMAGE_INTERVENTION_CSV = REPORT_DIR / 'phase8_image_concept_interventions_notebook.csv'
TCAV_HEATMAP_FIG = FIG_DIR / 'phase7_tcav_heatmap_notebook.png'
TCAV_EFFECT_FIG = FIG_DIR / 'phase7_tcav_top_scores_notebook.png'
CBM_SUMMARY_FIG = FIG_DIR / 'phase8_cbm_summary_notebook.png'
IMAGE_INTERVENTION_FIG = FIG_DIR / 'phase8_image_concept_interventions_notebook.png'
required_cbm_outputs = [CBM_SUMMARY_CSV, IMAGE_INTERVENTION_CSV, CBM_SUMMARY_FIG, IMAGE_INTERVENTION_FIG]
missing = [path for path in required_cbm_outputs if not path.exists()]
if missing:
    raise FileNotFoundError(f'Regenerate CBM outputs before blog export: {missing}')

tcav_available = TCAV_CSV.exists()
tcav = pd.read_csv(TCAV_CSV) if tcav_available else pd.DataFrame()
cbm = pd.read_csv(CBM_SUMMARY_CSV).iloc[0]
image_interventions = pd.read_csv(IMAGE_INTERVENTION_CSV)
validated_tcav_columns = {
    'concept', 'target_class', 'effect_size', 'tcav_std',
    'cav_validation_accuracy_mean', 'random_cav_validation_accuracy_mean',
    'p_value_corrected', 'significant',
}
legacy_tcav_columns = {'concept', 'target_class', 'tcav_score', 'cav_train_accuracy'}
tcav_is_validated = tcav_available and validated_tcav_columns.issubset(tcav.columns)

if tcav_is_validated:
    numeric_columns = [
        'effect_size', 'tcav_std', 'cav_validation_accuracy_mean',
        'random_cav_validation_accuracy_mean', 'p_value_corrected',
    ]
    for column in numeric_columns:
        tcav[column] = pd.to_numeric(tcav[column], errors='raise')
    tcav['significant_bool'] = tcav['significant'].astype(str).str.lower().eq('true')
    selected_tcav = tcav.sort_values(
        ['significant_bool', 'effect_size'], ascending=[False, False]
    ).head(10).copy()
elif tcav_available and legacy_tcav_columns.issubset(tcav.columns):
    missing_legacy = legacy_tcav_columns.difference(tcav.columns)
    if missing_legacy:
        raise ValueError(f'TCAV CSV has an unsupported schema. Missing: {sorted(missing_legacy)}')
    tcav['tcav_score'] = pd.to_numeric(tcav['tcav_score'], errors='raise')
    tcav['cav_train_accuracy'] = pd.to_numeric(tcav['cav_train_accuracy'], errors='raise')
    selected_tcav = tcav.sort_values('tcav_score', ascending=False).head(10).copy()
    print(
        'TCAV note: the available CSV is exploratory legacy output. The figure reports '
        'raw TCAV scores and CAV training accuracy only; it does not claim held-out '
        'validation, random-control effects, variability, or statistical significance.'
    )
else:
    tcav_available = False
    selected_tcav = pd.DataFrame({
        'concept': ['not available'],
        'target_class': [''],
        'tcav_score': [0.0],
        'cav_train_accuracy': [0.0],
    })
    print('TCAV outputs are unavailable; exporting explicit status panels without TCAV claims.')
selected_tcav['label'] = selected_tcav['concept'].astype(str) + ' -> ' + selected_tcav['target_class'].astype(str)

fig, axes = plt.subplots(1, 3, figsize=(16, 6.5), gridspec_kw={'width_ratios': [1.35, 1.1, 1.0]})
fig.patch.set_facecolor('white')
ax = axes[0]
positions = np.arange(len(selected_tcav))
if tcav_is_validated:
    colors = [PALETTE['teal'] if value else '#9aa7ad' for value in selected_tcav['significant_bool']]
    ax.barh(positions, selected_tcav['effect_size'], xerr=selected_tcav['tcav_std'], color=colors)
else:
    ax.barh(positions, selected_tcav['tcav_score'], color=PALETTE['teal'])
ax.set_yticks(positions, selected_tcav['label'])
ax.invert_yaxis()
if tcav_is_validated:
    ax.axvline(0, color=PALETTE['ink'], linewidth=0.8)
    ax.set_title('TCAV effect beyond random CAVs')
    ax.set_xlabel('mean real minus random score')
else:
    ax.set_xlim(0, 1)
    ax.set_title('Exploratory TCAV scores')
    ax.set_xlabel('fraction of positive directional derivatives')
ax.grid(axis='x', alpha=0.18)

ax = axes[1]
x_tcav = np.arange(len(selected_tcav))
if tcav_is_validated:
    width = 0.38
    ax.bar(x_tcav - width/2, selected_tcav['cav_validation_accuracy_mean'], width=width, color=PALETTE['teal'], label='real CAV')
    ax.bar(x_tcav + width/2, selected_tcav['random_cav_validation_accuracy_mean'], width=width, color=PALETTE['amber'], label='random CAV')
else:
    ax.bar(x_tcav, selected_tcav['cav_train_accuracy'], color=PALETTE['amber'])
ax.set_xticks(x_tcav, [str(value) for value in selected_tcav['concept']], rotation=55, ha='right')
ax.set_ylim(0, 1)
ax.set_title(
    'Held-out CAV validation'
    if tcav_is_validated
    else 'CAV fit on training activations\n(no held-out validation)'
)
ax.set_ylabel('accuracy')
if tcav_is_validated:
    ax.legend(frameon=False)
else:
    ax.set_xlabel('Training fit is not evidence of CAV generalization')
ax.grid(axis='y', alpha=0.18)

if not tcav_available:
    for status_ax, title, message in [
        (axes[0], 'TCAV results not available', 'No TCAV score CSV is present in the clean output state.'),
        (axes[1], 'CAV validation not available', 'No held-out CAV or random-control statistics are reported.'),
    ]:
        status_ax.clear()
        status_ax.axis('off')
        status_ax.text(0.5, 0.58, title, transform=status_ax.transAxes, ha='center', va='center', fontsize=14, fontweight='bold', color=PALETTE['navy'])
        status_ax.text(0.5, 0.40, message, transform=status_ax.transAxes, ha='center', va='center', color=PALETTE['muted'], wrap=True)

ax = axes[2]
original_wrong = ~image_interventions['original_correct'].astype(str).str.lower().eq('true')
corrected = image_interventions['intervened_correct'].astype(str).str.lower().eq('true')
per_image_corrected = corrected[original_wrong].groupby(image_interventions.loc[original_wrong, 'filepath']).max()
correction_rate = float(per_image_corrected.mean()) if len(per_image_corrected) else 0.0
cbm_labels = ['ResNet', 'CBM', 'AwA2 target injection']
cbm_values = [float(cbm['baseline_accuracy']), float(cbm['test_class_acc']), float(cbm['oracle_concept_class_accuracy'])]
bars = ax.bar(cbm_labels, cbm_values, color=[PALETTE['blue'], PALETTE['teal'], PALETTE['amber']])
ax.set_ylim(0, 1)
ax.set_title('CBM accuracy and target-injection stress test')
ax.set_ylabel('test accuracy')
ax.tick_params(axis='x', rotation=25)
for bar, value in zip(bars, cbm_values):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.025, f'{value:.2f}', ha='center', fontweight='bold')
ax.set_xlabel(
    f'Selected-error correction rate: {correction_rate:.1%}\n'
    'Target injection probes class-head distribution shift'
)
ax.grid(axis='y', alpha=0.18)

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
if tcav_is_validated:
    concept_figure_title = 'Validated concept evidence: CAV generalization, random controls and bottleneck decomposition'
elif tcav_available:
    concept_figure_title = 'Concept evidence: exploratory TCAV output and bottleneck decomposition'
else:
    concept_figure_title = 'Concept evidence: CBM results with TCAV status explicitly unavailable'
fig.suptitle(concept_figure_title, fontsize=15, fontweight='bold', color=PALETTE['navy'])
fig.tight_layout(rect=[0, 0, 1, 0.93])
out = FIG_DIR / 'blog_concept_validation_summary.png'
fig.savefig(out, bbox_inches='tight')
shutil.copy2(out, DOC_ASSET_DIR / 'concept-validation-summary.png')
if TCAV_HEATMAP_FIG.exists():
    shutil.copy2(TCAV_HEATMAP_FIG, DOC_ASSET_DIR / 'tcav-heatmap.png')
else:
    save_status_figure(DOC_ASSET_DIR / 'tcav-heatmap.png', 'TCAV heatmap not available', 'The current clean output state does not contain TCAV scores.')
if TCAV_EFFECT_FIG.exists():
    shutil.copy2(TCAV_EFFECT_FIG, DOC_ASSET_DIR / 'tcav-top-scores.png')
else:
    save_status_figure(DOC_ASSET_DIR / 'tcav-top-scores.png', 'TCAV effect plot not available', 'No validated or exploratory TCAV effect file is being reported.')
shutil.copy2(CBM_SUMMARY_FIG, DOC_ASSET_DIR / 'cbm-summary.png')
shutil.copy2(IMAGE_INTERVENTION_FIG, DOC_ASSET_DIR / 'concept-interventions.png')
plt.close(fig)
print(f'saved {out}')


saved /home/emma/DeepLearning/Deep_Learning_XAI/docs/blog_project_pipeline.svg
saved /home/emma/DeepLearning/Deep_Learning_XAI/docs/blog_method_taxonomy.svg
saved /home/emma/DeepLearning/Deep_Learning_XAI/docs/blog_evidence_ladder.svg
saved /home/emma/DeepLearning/Deep_Learning_XAI/outputs/figures/blog_metric_summary.png
saved /home/emma/DeepLearning/Deep_Learning_XAI/outputs/figures/blog_case_study_summary.png
saved /home/emma/DeepLearning/Deep_Learning_XAI/outputs/figures/blog_concept_validation_summary.png


Generated assets:

- `docs/blog_project_pipeline.svg`
- `docs/blog_method_taxonomy.svg`
- `docs/blog_evidence_ladder.svg`
- `outputs/figures/blog_metric_summary.png`
- `outputs/figures/blog_case_study_summary.png`
- `outputs/figures/blog_concept_validation_summary.png`
- `outputs/figures/xai_method_comparison_notebook.png`
- `docs/assets/xai-report/metric-summary.png`
- `docs/assets/xai-report/case-summary.png`
- `docs/assets/xai-report/concept-validation-summary.png`
- `docs/assets/xai-report/correct-attributions.png`
- refreshed `advanced-gradcam-deletion.png` and `advanced-ig-deletion.png` in `docs/assets/xai-report/`
- refreshed `tcav-heatmap.png`, `tcav-top-scores.png`, `cbm-summary.png`, and `concept-interventions.png` in `docs/assets/xai-report/`